# Without Feature Engineering

## Imports

In [ ]:
from pyexpat import features

from preprocessing import preprocess_data
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
import numpy as np
import joblib
from xgboost import XGBRegressor
from catboost import CatBoostRegressor



In [ ]:
df = pd.read_csv('../data/combined_air_weather_5_cities_features.csv')
# Check missing values
print("Missing Values Summary:")
print(df.isnull().sum())
print(df.columns.tolist())




## Evaluation Function

In [ ]:
def evaluate_aqi_model(df, y_actual, y_pred, cities,  origins, horizon_hours, model_name="Model"):
    actual = np.asarray(y_actual)
    predictions = np.asarray(y_pred)
    cities = np.asarray(cities)

    origins = pd.to_datetime( origins, utc=True)

    df = df.copy()

    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"],utc=True)

    mae = mean_absolute_error( actual, predictions)

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predictions
        )
    )

    r2 = r2_score(
        actual,
        predictions
    )

    print("\n" + "=" * 60)
    print(f"{model_name.upper()} PERFORMANCE")
    print("=" * 60)

    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")

    baseline_times = (origins - pd.Timedelta(hours=horizon_hours))

    baseline_df = pd.DataFrame({"city": cities,"baseline_time": baseline_times})

    lookup_df = df[
        ["city", "timestamp_utc", "us_aqi"]
    ].rename(
        columns={
            "timestamp_utc": "baseline_time",
            "us_aqi": "baseline_prediction"
        }
    )

    baseline_df = baseline_df.merge(
        lookup_df,
        on=["city", "baseline_time"],
        how="left"
    )

    baseline_predictions = (
        baseline_df["baseline_prediction"]
        .values
    )

    valid = ~np.isnan(baseline_predictions)

    baseline_mae = mean_absolute_error(
        actual[valid],
        baseline_predictions[valid]
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(
            actual[valid],
            baseline_predictions[valid]
        )
    )

    improvement = (
        (baseline_mae - mae)
        / baseline_mae
    ) * 100

    print("\n" + "=" * 60)
    print("PERSISTENCE BASELINE")
    print("=" * 60)

    print(f"Model MAE:       {mae:.2f}")
    print(f"Baseline MAE:    {baseline_mae:.2f}")

    print(f"Model RMSE:      {rmse:.2f}")
    print(f"Baseline RMSE:   {baseline_rmse:.2f}")

    print(f"MAE improvement: {improvement:.2f}%")


    print("\n" + "=" * 60)
    print("PER-CITY PERFORMANCE")
    print("=" * 60)
    city_results = []

    for city in np.unique(cities):

        mask = cities == city

        city_actual = actual[mask]
        city_pred = predictions[mask]

        city_mae = mean_absolute_error(
            city_actual,
            city_pred
        )

        city_rmse = np.sqrt(
            mean_squared_error(
                city_actual,
                city_pred
            )
        )

        city_r2 = r2_score(
            city_actual,
            city_pred
        )

        city_results.append({
            "city": city,
            "samples": mask.sum(),
            "MAE": city_mae,
            "RMSE": city_rmse,
            "R2": city_r2
        })

        print(
            f"{city}: "
            f"n={mask.sum()}, "
            f"MAE={city_mae:.2f}, "
            f"RMSE={city_rmse:.2f}, "
            f"R²={city_r2:.3f}"
        )

    city_results = pd.DataFrame(city_results)


    print("\n" + "=" * 60)
    print("PERFORMANCE BY AQI RANGE")
    print("=" * 60)

    bins = [0, 50,  100, 150, 200, 300, np.inf]

    labels = [
        "0-50",
        "51-100",
        "101-150",
        "151-200",
        "201-300",
        "300+"
    ]

    categories = pd.cut(
        actual,
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    range_results = []

    for category in labels:
        mask = categories == category
        if mask.sum() == 0:
            continue
        range_mae = mean_absolute_error( actual[mask],  predictions[mask])

        range_rmse = np.sqrt(
            mean_squared_error(
                actual[mask],
                predictions[mask]
            )
        )
        range_results.append({"AQI_range": category,"samples": mask.sum(), "MAE": range_mae,"RMSE": range_rmse})

        print(
            f"{category}: "
            f"n={mask.sum()}, "
            f"MAE={range_mae:.2f}, "
            f"RMSE={range_rmse:.2f}"
        )

    range_results = pd.DataFrame(range_results)

    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "baseline_MAE": baseline_mae,
        "baseline_RMSE": baseline_rmse,
        "MAE_improvement": improvement,
        "city_results": city_results,
        "aqi_range_results": range_results
    }

## Import Data

In [ ]:
filepath = '../data/combined_air_weather_5_cities_features.csv'

X_train, X_test, y_train, y_test, encoder, train, test, df_processed, test_cities, test_origins = preprocess_data(filepath, target_column="target_24h")

print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))
features = X_train.columns.tolist()
print(features)


## Training Linear Models

In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )}
results = []
model_predictions = {}

for name, l_model in models.items():

    print(f"\nTraining {name}")

    l_model.fit(X_train, y_train)

    predictions = l_model.predict(X_test)

    model_predictions[name] = predictions

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })
results = pd.DataFrame(results)

print("\nModel Performance")
print(results.sort_values("MAE"))

rf_stats = evaluate_aqi_model(
    df=df_processed,
    y_actual=y_test,
    y_pred=model_predictions["Random Forest"],
    cities=test_cities,
    origins=test_origins,
    horizon_hours=72,
    model_name="Random Forest"
)


lr_stats = evaluate_aqi_model(
    df=df_processed,
    y_actual=y_test,
    y_pred=model_predictions["Linear Regression"],
    cities=test_cities,
    origins=test_origins,
    horizon_hours=72,
    model_name="Linear Regression"
)

ridge_stats = evaluate_aqi_model(
    df=df_processed,
    y_actual=y_test,
    y_pred=model_predictions["Ridge Regression"],
    cities=test_cities,
    origins=test_origins,
    horizon_hours=72,
    model_name="Ridge Regression"
)

## Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Make predictions
rf_predictions = rf_model.predict(X_test)

# Calculate evaluation metrics
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

# Create results dataframe for Random Forest
rf_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "MAE": [rf_mae],
    "RMSE": [rf_rmse],
    "R2": [rf_r2]
})
joblib.dump(rf_model, "../models/rf_model_72h.pkl")

rf_stats = evaluate_aqi_model(df=df_processed, y_actual=y_test,  y_pred=rf_predictions, cities=test_cities, origins=test_origins, horizon_hours=72, model_name="Random Forest")



## Train XGBoost Regressor

In [ ]:

xg_model = XGBRegressor( n_estimators=300, learning_rate=0.05,max_depth=8, random_state=42)

xg_model.fit(X_train, y_train)
pred = xg_model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("R² :", r2_score(y_test, pred))
joblib.dump(xg_model, "../models/xgboost_24h.pkl")
xg_stats = evaluate_aqi_model(df=df_processed, y_actual=y_test, y_pred=pred, cities=test_cities, origins=test_origins, horizon_hours=24, model_name="XGboost")


## Train Catboost Regressor

In [ ]:

cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    verbose=False,
    random_seed=42
)

cat_model.fit(X_train, y_train)

cat_predictions = cat_model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, cat_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, cat_predictions)))
print("R²  :", r2_score(y_test, cat_predictions))


cat_model.save_model("../models/catboost_48h.cbm")
catboost_stats = evaluate_aqi_model(df=df_processed,y_actual=y_test, y_pred=cat_predictions,cities=test_cities,origins=test_origins,horizon_hours=24,model_name="CatBoost")



# Abalation Testing

## Individual Features Abalation testing

### Catboost (48hr model)

In [ ]:


filepath = '../data/combined_air_weather_5_cities_features.csv'

X_train, X_test, y_train, y_test, encoder, train, test, df_processed, test_cities, test_origins = preprocess_data(
    filepath,
    target_column="target_48h"
)

features_to_remove = [
    "is_rush_hour",
    "precipitation",
    "no2",
    "wind_speed",
    "hour",
    "is_weekend",
    "aqi_diff_24h",
    "wind_direction",
]


baseline_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    verbose=False,
    random_seed=42
)

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("=" * 60)
print("BASELINE — ALL FEATURES")
print("=" * 60)
print(f"MAE : {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²  : {baseline_r2:.4f}")

results = []

for feature in features_to_remove:

    X_train_reduced = X_train.drop(columns=[feature])
    X_test_reduced = X_test.drop(columns=[feature])

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        verbose=False,
        random_seed=42
    )

    model.fit(X_train_reduced, y_train)

    pred = model.predict(X_test_reduced)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    mae_change = mae - baseline_mae

    results.append({
        "Removed Feature": feature,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAE Change": mae_change,
        "Improved?": "YES" if mae < baseline_mae else "NO"
    })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values("MAE")

print("\n" + "=" * 80)
print("FEATURE ABLATION RESULTS")
print("=" * 80)

print(results_df.to_string(index=False))

print("\n" + "=" * 80)
print("BASELINE MAE:", round(baseline_mae, 4))
print("=" * 80)

### Random Forest (72hr model)

In [ ]:


filepath = '../data/combined_air_weather_5_cities_features.csv'

X_train, X_test, y_train, y_test, encoder, train, test, df_processed, test_cities, test_origins = preprocess_data(
    filepath,
    target_column="target_72h"
)

features_to_remove = [
    "is_rush_hour",
    "precipitation",
    "no2",
    "wind_speed",
    "hour",
    "is_weekend",
    "aqi_diff_24h",
    "wind_direction",
]


baseline_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("=" * 60)
print("RANDOM FOREST BASELINE")
print("=" * 60)
print(f"MAE : {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²  : {baseline_r2:.4f}")

results = []

for feature in features_to_remove:
    X_train_reduced = X_train.drop(columns=[feature])
    X_test_reduced = X_test.drop(columns=[feature])

    model = RandomForestRegressor(n_estimators=200, random_state=42,n_jobs=-1)

    model.fit(X_train_reduced, y_train)

    pred = model.predict(X_test_reduced)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Removed Feature": feature,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAE Change": mae - baseline_mae,
        "Improved?": "YES" if mae < baseline_mae else "NO"
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values("MAE").to_string(index=False))

### XGBoost (24hr model)

In [ ]:


filepath = '../data/combined_air_weather_5_cities_features.csv'

X_train, X_test, y_train, y_test, encoder, train, test, df_processed, test_cities, test_origins = preprocess_data(
    filepath,
    target_column="target_24h"
)

features_to_remove = [
    "is_rush_hour",
    "precipitation",
    "no2",
    "wind_speed",
    "hour",
    "is_weekend",
    "aqi_diff_24h",
    "wind_direction",
]

baseline_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("=" * 60)
print("XGBOOST BASELINE")
print("=" * 60)
print(f"MAE : {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²  : {baseline_r2:.4f}")

results = []

for feature in features_to_remove:

    X_train_reduced = X_train.drop(columns=[feature])
    X_test_reduced = X_test.drop(columns=[feature])

    model = XGBRegressor( n_estimators=500, learning_rate=0.05, max_depth=8, random_state=42, n_jobs=-1)

    model.fit(X_train_reduced, y_train)

    pred = model.predict(X_test_reduced)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Removed Feature": feature,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAE Change": mae - baseline_mae,
        "Improved?": "YES" if mae < baseline_mae else "NO"
    })


results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("XGBOOST FEATURE ABLATION")
print("=" * 80)

print(results_df.sort_values("MAE").to_string(index=False))

## Group Features Abalation Testing

### Catboost

In [ ]:
features_to_remove = [
    "aqi_diff_24h",
    "wind_speed",
    "is_rush_hour"
]
filepath = '../data/combined_air_weather_5_cities_features.csv'

X_train, X_test, y_train, y_test, encoder, train, test, df_processed, test_cities, test_origins = preprocess_data(
    filepath,
    target_column="target_48h"
)

X_train_reduced = X_train.drop(columns=features_to_remove)
X_test_reduced = X_test.drop(columns=features_to_remove)

cat_model_reduced = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    verbose=False,
    random_seed=42
)

cat_model_reduced.fit(X_train_reduced, y_train)
pred = cat_model_reduced.predict(X_test_reduced)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"Removed: {features_to_remove}")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

In [ ]:

filepath = '../data/combined_air_weather_5_cities_features.csv'

X_train, X_test, y_train, y_test, encoder, train, test, df_processed, test_cities, test_origins = preprocess_data(
    filepath,
    target_column="target_72h"
)
features_to_remove = [
    "aqi_diff_24h",
    "wind_speed",
    "is_rush_hour"
]

X_train_reduced = X_train.drop(columns=features_to_remove)
X_test_reduced = X_test.drop(columns=features_to_remove)

rf_model_reduced = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model_reduced.fit(X_train_reduced, y_train)

pred = rf_model_reduced.predict(X_test_reduced)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("RANDOM FOREST — AFTER FEATURE REMOVAL")
print("=" * 50)
print(f"Removed: {features_to_remove}")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

In [ ]:

filepath = '../data/combined_air_weather_5_cities_features.csv'
X_train, X_test, y_train, y_test, encoder, train, test, df_processed, test_cities, test_origins = preprocess_data(
    filepath,
    target_column="target_24h"
)
features_to_remove = [
    "aqi_diff_24h",
    "wind_speed",
    "is_rush_hour"
]

X_train_reduced = X_train.drop(columns=features_to_remove)
X_test_reduced = X_test.drop(columns=features_to_remove)

xgb_model_reduced = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

xgb_model_reduced.fit(X_train_reduced, y_train)

pred = xgb_model_reduced.predict(X_test_reduced)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
print(f"Removed: {features_to_remove}")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")